# Sherlock Holmes en Buenos Aires — Reel IA (50 s)

Pipeline pensado para **Google Colab + GPU**, usando **Wan 2.1 T2V 1.3B** en vertical nativo `480x832`.

Genera 10 planos de ~5 s, crea la voz en off exacta en español rioplatense, agrega overlays de deducción desde el segundo 10 y monta un Reel final de 50 s.

**Importante:** la continuidad absoluta del rostro entre planos no está garantizada por un modelo T2V. Se usa una descripción canónica idéntica + seeds deterministas para reducir drift.

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.


In [ ]:
# 1) Preparación. Evitamos flash-attn para no compilar cosas innecesarias en Colab.
import os, sys, subprocess, shutil
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['rm','-rf','/content/Wan2.1'], check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/Wan-Video/Wan2.1.git','/content/Wan2.1'], check=True)
pkgs=['opencv-python>=4.9.0.80','diffusers>=0.31.0','transformers>=4.49.0','tokenizers>=0.20.3','accelerate>=1.1.1','tqdm','imageio','easydict','ftfy','dashscope','imageio-ffmpeg','huggingface_hub','edge-tts']
subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs], check=True)
print('✅ Entorno listo')


In [ ]:
# 2) Guardado persistente en Google Drive (MUY recomendado).
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/Sherlock_Buenos_Aires_Reel')
RAW=ROOT/'raw'; FINAL=ROOT/'final'; AUDIO=ROOT/'audio'
for p in (RAW,FINAL,AUDIO): p.mkdir(parents=True, exist_ok=True)
print('✅ Proyecto:', ROOT)


In [ ]:
# 3) Descargar Wan 2.1 1.3B una sola vez al Drive. Puede tardar.
from huggingface_hub import snapshot_download
MODEL_DIR=ROOT/'Wan2.1-T2V-1.3B'
if not (MODEL_DIR/'config.json').exists():
    print('Descargando Wan 2.1 T2V 1.3B...')
    snapshot_download('Wan-AI/Wan2.1-T2V-1.3B', local_dir=str(MODEL_DIR))
else:
    print('✅ Modelo ya estaba descargado')
print(MODEL_DIR)


## Guion y planos
Cada plano dura exactamente 5 segundos en el montaje. El prompt visual está en inglés porque estos modelos suelen seguirlo mejor; la voz permanece en español.


In [ ]:
# 4) Guion de producción.
CHARACTER = '''Sherlock Holmes, a fictional British consulting detective with classic Victorian aesthetics, NOT resembling any known actor: tall slim man in his early forties, pale angular intellectual face, dark slightly wavy hair, sharp observant grey eyes, long dark charcoal overcoat, elegant dark three-piece suit, white shirt, waistcoat, black leather boots. The exact same character design in every shot.'''
STYLE = '''hyperrealistic cinematic prestige detective film, contemporary Buenos Aires, natural human motion, realistic skin, realistic crowds, subtle volumetric city light, premium cinematography, elegant camera movement, shallow depth of field, centered portrait-friendly composition, no cyberpunk, no steampunk exaggeration, no videogame HUD, no readable private data, no celebrity likeness'''
NEG = 'cartoon, anime, illustration, cyberpunk, steampunk machinery, neon overload, videogame HUD, malformed hands, extra fingers, duplicated people, warped faces, distorted Obelisk, American city, New York, London, unreadable text, watermark, logo, celebrity actor likeness'
SCENES=[
 {'id':1,'seed':1901,'ar':False,'voice':'Esto no es Londres. No. Ah, claro... Buenos Aires.','prompt':'Sherlock has just appeared unexpectedly at Avenida 9 de Julio and Avenida Corrientes. He walks slowly among modern pedestrians, genuinely shocked and disoriented. The Obelisk is clearly visible behind him. Contemporary Argentine traffic and Buenos Aires architecture. Smooth medium tracking shot.'},
 {'id':2,'seed':1902,'ar':False,'voice':'Demasiado ruido. Demasiada luz. Demasiada velocidad. ¿Y ese monumento...? No importa.','prompt':'Sherlock studies the modern city around the Buenos Aires Obelisk: buses, taxis, cars, bright Corrientes signs, smartphones and pedestrians. POV details alternate with a close-up of his puzzled face. He looks upward at the Obelisk. No augmented reality yet.'},
 {'id':3,'seed':1903,'ar':True,'voice':'Observemos.','prompt':'Close-up on Sherlock beside Avenida Corrientes. On the word observe his expression changes from confusion to total cold analytical concentration. He notices a man in a grey business suit nearby holding a smartphone. Elegant slow push-in, then Sherlock gaze POV.'},
 {'id':4,'seed':1904,'ar':True,'voice':'El hombre del traje gris lleva doce minutos esperando. Mira el teléfono cada diecisiete segundos. No espera un taxi. Espera una respuesta.','prompt':'Sherlock discreetly analyzes a man wearing a grey suit near the intersection. The man repeatedly checks his phone, shifts posture, glances around. Sherlock watches silently from mid distance. Sophisticated detective cinematography, visual emphasis on tiny behavioral clues.'},
 {'id':5,'seed':1905,'ar':True,'voice':'La mujer junto al semáforo viene de una oficina cercana. Café sin terminar, credencial todavía colgada, pasos rápidos... llega tarde.','prompt':'Sherlock observes a woman walking quickly beside a pedestrian traffic light in Buenos Aires. She carries an unfinished takeaway coffee and a visible generic office badge, walking rapidly with purposeful body language. POV inserts of coffee, badge and steps.'},
 {'id':6,'seed':1906,'ar':True,'voice':'Ese colectivo frenará antes de tiempo. El conductor ya vio al peatón.','prompt':'A recognizable contemporary Buenos Aires city bus approaches the intersection while a pedestrian nears the crossing. Sherlock watches and calmly predicts the motion. The bus brakes safely slightly early. Realistic Argentine traffic, no crash, cinematic telephoto shot.'},
 {'id':7,'seed':1907,'ar':True,'voice':'Y entonces... están ellos. Los teléfonos. Ubicación. Fotografías. Conversaciones. Compras.','prompt':'Sherlock suddenly notices that almost everyone around the Obelisk is carrying a smartphone. Elegant close-ups of different generic phones in many hands, then Sherlock fascinated by the abundance of clues. Crowd remains natural and realistic.'},
 {'id':8,'seed':1908,'ar':True,'voice':'Rutinas. Preferencias. Dónde estuvieron. Con quién. A qué hora. Qué desean. Qué temen.','prompt':'Wide cinematic shot of a modern Buenos Aires crowd around Sherlock. His analytical mind recognizes abstract patterns linking phones, movements, habits and schedules. The visual idea becomes denser but remains elegant and restrained, never cyberpunk.'},
 {'id':9,'seed':1909,'ar':True,'voice':'En mi época, un hombre podía pasar toda una vida intentando ocultar sus secretos. Ahora los llevan en el bolsillo... y los revelan voluntariamente. Todo el tiempo.','prompt':'Sherlock walks slowly through the crowd with complete confidence, seeing an immense abstract network of clues around ordinary people and their phones. People continue their normal lives, unaware. Premium detective-film atmosphere, subtle amused arrogance.'},
 {'id':10,'seed':1910,'ar':True,'voice':'Una ciudad entera dejando un rastro perfecto. Esto va a ser... muy fácil.','prompt':'Final hero shot in Avenida Corrientes with the Obelisk and Buenos Aires life behind Sherlock. He stops, looks directly past the camera, the mental network of clues organizes instantly. Very subtle smile. Extreme close-up of his observant eyes, then dramatic cinematic finish.'}
]
for s in SCENES: s['full_prompt']=f"{CHARACTER} {s['prompt']} {STYLE}"
print('✅',len(SCENES),'planos · 50 segundos finales')


In [ ]:
# 5) Generar voz exacta por plano. Se acelera automáticamente si una frase supera 4.85 s.
import asyncio, subprocess, json, math, os
VOICE='es-AR-TomasNeural'
RATE='+25%'
def duration(path):
    r=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],capture_output=True,text=True,check=True)
    return float(r.stdout.strip())
async def make_tts(text,path):
    import edge_tts
    await edge_tts.Communicate(text,VOICE,rate=RATE).save(str(path))
for s in SCENES:
    p=AUDIO/f"voice_{s['id']:02d}.mp3"
    if not p.exists(): asyncio.run(make_tts(s['voice'],p))
    d=duration(p)
    if d>4.85:
        factor=d/4.85
        tmp=p.with_name(p.stem+'_fit.mp3')
        subprocess.run(['ffmpeg','-y','-i',str(p),'-filter:a',f'atempo={factor:.5f}',str(tmp)],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL,check=True)
        tmp.replace(p); d=duration(p)
    print(f"Plano {s['id']:02d}: voz {d:.2f}s")
print('✅ Voz lista')


## Render
Para probar primero, dejá `SCENES_TO_RENDER=[1]`. Cuando veas que funciona, cambialo a `list(range(1,11))`. Los planos existentes se saltean automáticamente.


In [ ]:
# 6) Elegí qué planos generar. Primera prueba: sólo plano 1.
SCENES_TO_RENDER=[1]  # luego: list(range(1,11))
SAMPLE_STEPS=30       # 30 para pruebas; 50 para versión final
print('Render:',SCENES_TO_RENDER)


In [ ]:
# 7) Render Wan 2.1. Cada plano se guarda inmediatamente en Drive.
import subprocess, os, gc, torch
for s in SCENES:
    if s['id'] not in SCENES_TO_RENDER: continue
    out=RAW/f"scene_{s['id']:02d}.mp4"
    if out.exists() and out.stat().st_size>100000:
        print('⏭️ Ya existe',out.name); continue
    print(f"🎬 Render plano {s['id']:02d}...")
    cmd=[sys.executable,'/content/Wan2.1/generate.py','--task','t2v-1.3B','--size','480*832','--ckpt_dir',str(MODEL_DIR),'--offload_model','True','--t5_cpu','--sample_shift','8','--sample_guide_scale','6','--sample_steps',str(SAMPLE_STEPS),'--base_seed',str(s['seed']),'--prompt',s['full_prompt'],'--save_file',str(out)]
    subprocess.run(cmd,cwd='/content/Wan2.1',check=True)
    torch.cuda.empty_cache(); gc.collect()
    print('✅',out)


In [ ]:
# 8) Post: duración exacta 5s, HUD mental sutil desde el segundo 10 y voz.
import cv2, numpy as np, subprocess, os, math
def hud_video(src,dst):
    cap=cv2.VideoCapture(str(src)); fps=cap.get(cv2.CAP_PROP_FPS) or 16
    w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc=cv2.VideoWriter_fourcc(*'mp4v'); wr=cv2.VideoWriter(str(dst),fourcc,fps,(w,h))
    i=0
    while True:
        ok,fr=cap.read()
        if not ok: break
        t=i/fps; layer=fr.copy()
        pts=[(int(w*.28+10*math.sin(t)),int(h*.38)),(int(w*.70),int(h*.52+8*math.cos(t*.8))),(int(w*.48),int(h*.70))]
        for k,(x,y) in enumerate(pts):
            r=24+((i+k*7)%12)
            cv2.rectangle(layer,(x-r,y-r),(x+r,y+r),(235,245,245),1)
            cv2.line(layer,(x+r,y),(min(w-1,x+r+45),y),(235,245,245),1)
        cv2.line(layer,pts[0],pts[1],(220,235,235),1); cv2.line(layer,pts[1],pts[2],(220,235,235),1)
        fr=cv2.addWeighted(layer,0.28,fr,0.72,0)
        wr.write(fr); i+=1
    wr.release(); cap.release()
for s in SCENES:
    src=RAW/f"scene_{s['id']:02d}.mp4"
    if not src.exists(): continue
    visual=FINAL/f"visual_{s['id']:02d}.mp4"
    if s['ar']:
        tmp=FINAL/f"hud_{s['id']:02d}.mp4"; hud_video(src,tmp); use=tmp
    else: use=src
    # retime/trim a 5 segundos exactos, manteniendo 480x832 vertical
    d=duration(use); ratio=5.0/d
    subprocess.run(['ffmpeg','-y','-i',str(use),'-vf',f'setpts={ratio:.8f}*PTS,fps=30,scale=1080:1920:flags=lanczos','-an','-t','5',str(visual)],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL,check=True)
    out=FINAL/f"scene_{s['id']:02d}_final.mp4"; voice=AUDIO/f"voice_{s['id']:02d}.mp3"
    subprocess.run(['ffmpeg','-y','-i',str(visual),'-i',str(voice),'-filter_complex','[1:a]apad=pad_dur=5[a]','-map','0:v','-map','[a]','-c:v','copy','-c:a','aac','-b:a','192k','-t','5',str(out)],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL,check=True)
    print('✅ post',s['id'])


In [ ]:
# 9) Montaje final: requiere los 10 planos.
missing=[s['id'] for s in SCENES if not (FINAL/f"scene_{s['id']:02d}_final.mp4").exists()]
if missing:
    print('⚠️ Faltan planos:',missing,'— generá esos planos y repetí post + montaje.')
else:
    concat=ROOT/'concat.txt'
    concat.write_text(''.join([f"file '{FINAL/f'scene_{i:02d}_final.mp4'}'\n" for i in range(1,11)]))
    silent=FINAL/'sherlock_buenos_aires_50s.mp4'
    subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(concat),'-c','copy',str(silent)],check=True)
    print('🎉 REEL LISTO:',silent)
    from google.colab import files
    files.download(str(silent))


## Uso recomendado
1. Ejecutá todo hasta la celda **6**.
2. Renderizá `SCENES_TO_RENDER=[1]` para comprobar que Wan funciona en tu GPU.
3. Si quedó bien, usá `SCENES_TO_RENDER=list(range(1,11))`. Los planos ya existentes se saltean.
4. Para el render definitivo cambiá `SAMPLE_STEPS=50`.
5. Ejecutá Post y Montaje.

Los archivos quedan persistidos en `Mi unidad/Sherlock_Buenos_Aires_Reel`, así que una desconexión de Colab no borra lo ya generado.
